In [1]:
##%%
# ============================================
# IMPORTS AND CONFIGURATION
# ============================================
import os
import sys
import warnings
warnings.filterwarnings('ignore')

# Force legacy Keras for TensorFlow Probability compatibility
os.environ["TF_USE_LEGACY_KERAS"] = "1"

import numpy as np
import polars as pl
import pandas as pd
import lightgbm as lgb
import tensorflow as tf
import tensorflow_probability as tfp
from tensorflow.keras import layers, models, callbacks
import datetime
import time
from pathlib import Path

tfd = tfp.distributions

# Check GPU availability
gpus = tf.config.list_physical_devices('GPU')
if gpus:
    print(f"✅ GPU Available: {gpus[0]}")
else:
    print("⚠️ GPU not available")
print("="*60)

# Polars configuration
pl.Config.set_tbl_rows(100)
pl.Config.set_tbl_cols(100)

# ============================================
# VERSION CHECK (required by competition rules)
# ============================================
print("="*60)
print("ENVIRONMENT VERSIONS")
print("="*60)
print(f"Python version: {sys.version}")
print(f"LightGBM version: {lgb.__version__}")
print(f"Polars version: {pl.__version__}")
print(f"NumPy version: {np.__version__}")
print("="*60)

pl.Config.set_tbl_rows(100)
pl.Config.set_tbl_cols(100)

2026-04-07 14:05:39.991778: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:467] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1775570740.172694      24 cuda_dnn.cc:8579] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1775570740.229800      24 cuda_blas.cc:1407] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
W0000 00:00:1775570740.667448      24 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1775570740.667492      24 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1775570740.667496      24 computation_placer.cc:177] computation placer alr

✅ GPU Available: PhysicalDevice(name='/physical_device:GPU:0', device_type='GPU')
ENVIRONMENT VERSIONS
Python version: 3.12.12 (main, Oct 10 2025, 08:52:57) [GCC 11.4.0]
LightGBM version: 4.6.0
Polars version: 1.35.2
NumPy version: 2.0.2


polars.config.Config

In [2]:
#%%
##%%
# ============================================
# ENVIRONMENT SETUP
# ============================================
KAGGLE_ENV = os.path.exists('/kaggle/input')
if KAGGLE_ENV:
    train_path = '/kaggle/input/competitions/ts-forecasting/train.parquet'
    test_path = '/kaggle/input/competitions/ts-forecasting/test.parquet'
    output_dir = '/kaggle/working/'
else:
    base_dir = Path("..")
    train_path = base_dir / "data" / "train.parquet"
    test_path = base_dir / "data" / "test.parquet"
    output_dir = base_dir / "Results"
    output_dir.mkdir(parents=True, exist_ok=True)

print("✅ Environment ready")

# ============================================
# DATA LOADING
# ============================================
print("="*60)
print("LOADING DATA")
print("="*60)
train_full = pl.read_parquet(train_path)
test_full = pl.read_parquet(test_path)
print(f"Train: {train_full.shape}, Test: {test_full.shape}")
#%%
##%%
# ============================================
# SAFE IMPUTATION (no leakage)
# ============================================
print("\n" + "="*60)
print("SAFE IMPUTATION")
print("="*60)

# Focus on feature_a and ultra features for BNN
bnn_features = ['feature_al'] + ['feature_bz', 'feature_aw', 'feature_cc']
print(f"BNN features: {bnn_features}")

# Cross-horizon fill (same ts_index, different horizon)
group_keys_cross = ['code', 'sub_code', 'sub_category', 'ts_index']
for c in bnn_features:
    if train_full[c].null_count() > 0:
        train_full = train_full.with_columns(pl.col(c).fill_null(pl.col(c).first().over(group_keys_cross)).alias(c))
    if c in test_full.columns and test_full[c].null_count() > 0:
        test_full = test_full.with_columns(pl.col(c).fill_null(pl.col(c).first().over(group_keys_cross)).alias(c))

# Forward fill within group + horizon
group_keys_ff = ['code', 'sub_code', 'sub_category', 'horizon']
for c in bnn_features:
    if train_full[c].null_count() > 0:
        train_full = train_full.with_columns(pl.col(c).forward_fill().over(group_keys_ff).alias(c))
    if c in test_full.columns and test_full[c].null_count() > 0:
        test_full = test_full.with_columns(pl.col(c).forward_fill().over(group_keys_ff).alias(c))

# Causal imputation for remaining NaNs
causal_cols = [c for c in bnn_features if train_full[c].null_count() > 0 or (c in test_full.columns and test_full[c].null_count() > 0)]
group_keys = ['code', 'sub_code', 'sub_category', 'horizon']

if causal_cols:
    print(f"Processing {len(causal_cols)} features with remaining NaNs...")
    for c in causal_cols:
        print(f"   {c}")
        
        train_full = train_full.with_columns(
            pl.when(pl.col(c).is_null())
            .then(
                pl.coalesce([
                    pl.col(c).ewm_mean(span=20, adjust=False).over(group_keys),
                    pl.col(c).forward_fill().over(group_keys),
                    pl.col(c).ewm_mean(span=10, adjust=False).over(group_keys),
                    pl.lit(0.0)
                ])
            )
            .otherwise(pl.col(c))
            .alias(c)
        )
        
        if c in test_full.columns:
            test_full = test_full.with_columns(
                pl.when(pl.col(c).is_null())
                .then(
                    pl.coalesce([
                        pl.col(c).ewm_mean(span=20, adjust=False).over(group_keys),
                        pl.col(c).forward_fill().over(group_keys),
                        pl.col(c).ewm_mean(span=10, adjust=False).over(group_keys),
                        pl.lit(0.0)
                    ])
                )
                .otherwise(pl.col(c))
                .alias(c)
            )
    print("✅ Causal imputation complete (NO LEAKAGE)")
else:
    print("✅ No remaining NaNs - causal imputation skipped")

# Final fallback to 0
for c in bnn_features:
    if train_full[c].null_count() > 0:
        train_full = train_full.with_columns(pl.col(c).fill_null(0).alias(c))
    if c in test_full.columns and test_full[c].null_count() > 0:
        test_full = test_full.with_columns(pl.col(c).fill_null(0).alias(c))

print("Imputation done. NaNs: 0")
#%%
##%%
# ============================================
# TYPE CONVERSION (memory optimization)
# ============================================
print("\n" + "="*60)
print("TYPE CONVERSION")
print("="*60)

cat_cols = ['code', 'sub_code', 'sub_category']
for col in cat_cols:
    train_full = train_full.with_columns(pl.col(col).cast(pl.Categorical))
    test_full = test_full.with_columns(pl.col(col).cast(pl.Categorical))

int_cols = ['horizon', 'ts_index']
for col in int_cols:
    train_full = train_full.with_columns(pl.col(col).cast(pl.Int16))
    test_full = test_full.with_columns(pl.col(col).cast(pl.Int16))

# Convert BNN features to float32
for col in bnn_features:
    if train_full[col].dtype == pl.Float64:
        train_full = train_full.with_columns(pl.col(col).cast(pl.Float32))
    if test_full[col].dtype == pl.Float64:
        test_full = test_full.with_columns(pl.col(col).cast(pl.Float32))

print(f"BNN features: {len(bnn_features)}")
#%%
##%%
# ============================================
# FEATURE A + ULTRA FEATURES STANDARDIZATION
# ============================================
print("\n" + "="*60)
print("FEATURE A + ULTRA FEATURES STANDARDIZATION")
print("="*60)

def standardize_feature_with_weight(df: pl.DataFrame, features: list, group_cols: list, weights: list) -> pl.DataFrame:
    """Standardize features with different weights"""
    for feat, weight in zip(features, weights):
        if feat not in df.columns:
            continue
            
        # Apply weight
        df = df.with_columns(
            (pl.col(feat) * weight).alias(feat)
        )
        
        # Standardize
        df = df.with_columns(
            pl.when(pl.col(feat).std().over(group_cols) == 0)
            .then(0.0)
            .otherwise(
                ((pl.col(feat) - pl.col(feat).mean().over(group_cols)) /
                 (pl.col(feat).std().over(group_cols) + 1e-6))
            ).alias(feat)
        )
    return df

# Feature weights: feature_a = 1.0, ultra features = 0.2
feature_weights = [1.0, 0.05, 0.05, 0.05]  # feature_a + 3 ultra features

group_cols = ['code', 'sub_code', 'sub_category', 'horizon']

print("\n📊 Applying weighted standardization...")
train_full = standardize_feature_with_weight(train_full, bnn_features, group_cols, feature_weights)
test_full = standardize_feature_with_weight(test_full, bnn_features, group_cols, feature_weights)
print(f"  ✅ Weighted standardization complete: {len(bnn_features)} features")

# Debug check
print("\n📋 Feature ranges after standardization:")
for feat in bnn_features:
    if feat in train_full.columns:
        data = train_full.select(feat).to_numpy()
        nan_count = np.sum(np.isnan(data))
        inf_count = np.sum(np.isinf(data))
        min_val = np.min(data)
        max_val = np.max(data)
        weight = feature_weights[bnn_features.index(feat)]
        print(f"  {feat} (weight={weight}): NaN={nan_count}, Inf={inf_count}, range=[{min_val:.3f}, {max_val:.3f}]")
#%%
##%%
# ============================================
# CLEAN NaN/Inf AFTER STANDARDIZATION
# ============================================
print("\n🧹 Cleaning NaN/Inf after standardization...")

for col in bnn_features:
    # Clean train
    train_full = train_full.with_columns(
        pl.when(pl.col(col).is_null() | pl.col(col).is_nan() | pl.col(col).is_infinite())
        .then(0.0)
        .otherwise(pl.col(col))
        .alias(col)
    )
    
    # Clean test
    test_full = test_full.with_columns(
        pl.when(pl.col(col).is_null() | pl.col(col).is_nan() | pl.col(col).is_infinite())
        .then(0.0)
        .otherwise(pl.col(col))
        .alias(col)
    )

print("✅ All NaN/Inf in BNN features replaced with 0.0")

# Final verification
print("\n📋 Final verification:")
for feat in bnn_features:
    if feat in train_full.columns:
        data = train_full.select(feat).to_numpy()
        nan_count = np.sum(np.isnan(data))
        inf_count = np.sum(np.isinf(data))
        min_val = np.min(data)
        max_val = np.max(data)
        print(f"  {feat}: NaN={nan_count}, Inf={inf_count}, range=[{min_val:.3f}, {max_val:.3f}]")
#%%

✅ Environment ready
LOADING DATA
Train: (5337414, 94), Test: (1447107, 92)

SAFE IMPUTATION
BNN features: ['feature_al', 'feature_bz', 'feature_aw', 'feature_cc']
Processing 4 features with remaining NaNs...
   feature_al
   feature_bz
   feature_aw
   feature_cc
✅ Causal imputation complete (NO LEAKAGE)
Imputation done. NaNs: 0

TYPE CONVERSION
BNN features: 4

FEATURE A + ULTRA FEATURES STANDARDIZATION

📊 Applying weighted standardization...
  ✅ Weighted standardization complete: 4 features

📋 Feature ranges after standardization:
  feature_al (weight=1.0): NaN=33, Inf=0, range=[nan, nan]
  feature_bz (weight=0.05): NaN=33, Inf=0, range=[nan, nan]
  feature_aw (weight=0.05): NaN=33, Inf=0, range=[nan, nan]
  feature_cc (weight=0.05): NaN=33, Inf=0, range=[nan, nan]

🧹 Cleaning NaN/Inf after standardization...
✅ All NaN/Inf in BNN features replaced with 0.0

📋 Final verification:
  feature_al: NaN=0, Inf=0, range=[-13.488, 14.104]
  feature_bz: NaN=0, Inf=0, range=[-12.728, 6.708]
  f

In [3]:
# ============================================
# CLEAN NaN/Inf AFTER STANDARDIZATION
# ============================================
print("\n🧹 Cleaning NaN/Inf after standardization...")

for col in bnn_features:
    # Clean train
    train_full = train_full.with_columns(
        pl.when(pl.col(col).is_null() | pl.col(col).is_nan() | pl.col(col).is_infinite())
        .then(0.0)
        .otherwise(pl.col(col))
        .alias(col)
    )
    
    # Clean test
    test_full = test_full.with_columns(
        pl.when(pl.col(col).is_null() | pl.col(col).is_nan() | pl.col(col).is_infinite())
        .then(0.0)
        .otherwise(pl.col(col))
        .alias(col)
    )

print("✅ All NaN/Inf in BNN features replaced with 0.0")

# Final verification
print("\n📋 Final verification:")
for feat in bnn_features:
    if feat in train_full.columns:
        data = train_full.select(feat).to_numpy()
        nan_count = np.sum(np.isnan(data))
        inf_count = np.sum(np.isinf(data))
        min_val = np.min(data)
        max_val = np.max(data)
        print(f"  {feat}: NaN={nan_count}, Inf={inf_count}, range=[{min_val:.3f}, {max_val:.3f}]")
#%%


🧹 Cleaning NaN/Inf after standardization...
✅ All NaN/Inf in BNN features replaced with 0.0

📋 Final verification:
  feature_al: NaN=0, Inf=0, range=[-13.488, 14.104]
  feature_bz: NaN=0, Inf=0, range=[-12.728, 6.708]
  feature_aw: NaN=0, Inf=0, range=[-3.320, 13.428]
  feature_cc: NaN=0, Inf=0, range=[-13.065, 5.036]


In [4]:
##%%
# ============================================
# BNN MODEL - FEATURE A + ULTRA FEATURES
# ============================================

from tensorflow.keras import layers, models, callbacks
import tensorflow as tf
import tensorflow_probability as tfp

tfd = tfp.distributions

def build_bnn_model(window, n_features):
    inputs = layers.Input(shape=(window, n_features))
    
    x = layers.GlobalAveragePooling1D()(inputs)
    x = layers.Dense(64, activation='relu', kernel_initializer='he_normal')(x)
    x = layers.Dropout(0.1)(x)
    x = layers.Dense(32, activation='relu', kernel_initializer='he_normal')(x)
    x = layers.Dropout(0.1)(x)
    params = layers.Dense(2)(x)

    mu = layers.Lambda(lambda p: p[..., 0:1])(params)
    log_scale = layers.Lambda(lambda p: p[..., 1:2])(params)
    scale = layers.Lambda(lambda s: tf.nn.softplus(s))(log_scale)

    outputs = layers.Lambda(lambda args: tf.concat(args, axis=-1))([mu, scale])
    model = models.Model(inputs=inputs, outputs=outputs)
    return model

def nll(y_true, y_pred):
    mu = y_pred[..., 0:1]
    scale = y_pred[..., 1:2]
    scale = tf.maximum(scale, 1e-6)  # prevent NaN
    dist = tfd.Laplace(loc=mu, scale=scale)
    return -dist.log_prob(y_true)
##%%
##%%
# ============================================
# BNN TRAINING PER HORIZON
# ============================================

horizons = [1, 3, 10, 25]
window_size = {1: 10, 3: 15, 10: 20, 25: 25}
EPOCHS = 20

# Add BNN columns
for h in horizons:
    for suffix in ['mean', 'scale']:
        col_name = f'bnn_{suffix}_h{h}'
        if col_name not in train_full.columns:
            train_full = train_full.with_columns(pl.lit(0.0).alias(col_name))
        if col_name not in test_full.columns:
            test_full = test_full.with_columns(pl.lit(0.0).alias(col_name))

print("\n" + "="*60)
print("BNN TRAINING - FEATURE A + ULTRA FEATURES")
print("="*60)

for h in horizons:
    print(f"\n--- HORIZON {h} ---")
    window = window_size[h]

    train_h = train_full.filter(pl.col('horizon') == h).sort('ts_index')

    if len(train_h) == 0:
        print(f"  ⚠️ No data for H={h}. Skipping BNN.")
        continue

    # Use only feature_a + ultra features
    X_full = train_h.select(bnn_features).to_numpy().astype(np.float32)
    y_full = train_h['y_target'].to_numpy().astype(np.float32)
    
    y_mean = np.mean(y_full)
    y_std = np.std(y_full)
    y_full_standardized = (y_full - y_mean) / y_std
    print(f"  Target: mean={y_mean:.3f}, std={y_std:.3f}")
    
    if len(X_full) <= window:
        print(f"  ⚠️ Not enough data for H={h} (len={len(X_full)} <= window={window}). Skipping BNN.")
        continue

    # Create sliding windows
    X_windows, y_windows = [], []
    for i in range(len(X_full) - window):
        X_windows.append(X_full[i:i+window])
        y_windows.append(y_full_standardized[i+window])

    X_train = np.array(X_windows)
    y_train = np.array(y_windows)

    # Debug
    print(f"\n🔍 DEBUG BNN TRAINING DATA H={h}:")
    print(f"  X_train shape: {X_train.shape}")
    print(f"  y_train shape: {y_train.shape}")
    print(f"  X_train min/max: [{np.min(X_train):.6f}, {np.max(X_train):.6f}]")
    print(f"  y_train min/max: [{np.min(y_train):.6f}, {np.max(y_train):.6f}]")
    print(f"  X_train NaN count: {np.sum(np.isnan(X_train))}")
    print(f"  y_train NaN count: {np.sum(np.isnan(y_train))}")

    n_features = X_train.shape[2]
    model = build_bnn_model(window, n_features)

    model.compile(optimizer=tf.keras.optimizers.Adam(0.001), loss=nll)
    early_stop = callbacks.EarlyStopping(monitor='loss', patience=5, restore_best_weights=True)
    
    model.fit(X_train, y_train, batch_size=128, epochs=EPOCHS, verbose=1, callbacks=[early_stop])

    # Predictions for training
    X_full_all = train_h.select(bnn_features).to_numpy().astype(np.float32)
    n_samples = len(X_full_all)
    X_windows_all = []
    for i in range(n_samples - window):
        X_windows_all.append(X_full_all[i:i+window])
    X_windows_all = np.array(X_windows_all)

    pred = model.predict(X_windows_all)
    mean_pred = pred[..., 0].flatten()
    scale_pred = pred[..., 1].flatten()
    
    mean_pred = mean_pred * y_std + y_mean
    print(f"  Destandaryzowane mean range: [{np.min(mean_pred):.6f}, {np.max(mean_pred):.6f}]")
    print(f"\n🔍 DEBUG BNN PREDICTIONS H={h}:")
    print(f"  Mean range: [{np.min(mean_pred):.6f}, {np.max(mean_pred):.6f}]")
    print(f"  Scale range: [{np.min(scale_pred):.6f}, {np.max(scale_pred):.6f}]")
    print(f"  Mean std: {np.std(mean_pred):.6f}")
    print(f"  Scale std: {np.std(scale_pred):.6f}")

    # Update train data
    full_mean = np.full(len(train_h), 0.0, dtype=np.float64)
    full_scale = np.full(len(train_h), 0.0, dtype=np.float64)
    full_mean[window:] = mean_pred.astype(np.float64)
    full_scale[window:] = scale_pred.astype(np.float64)

    train_h_updated = train_h.with_columns([
        pl.Series(f'bnn_mean_h{h}', full_mean).cast(pl.Float64),
        pl.Series(f'bnn_scale_h{h}', full_scale).cast(pl.Float64)
    ])
    
    train_full = train_full.filter(pl.col('horizon') != h)
    train_full = pl.concat([train_full, train_h_updated])

    # Predictions for test
    test_h = test_full.filter(pl.col('horizon') == h).sort('ts_index')
    X_test_all = test_h.select(bnn_features).to_numpy().astype(np.float64)
    n_test = len(X_test_all)

    if n_test > window:
        X_windows_test = []
        for i in range(n_test - window):
            X_windows_test.append(X_test_all[i:i+window])
        X_windows_test = np.array(X_windows_test)

        pred_test = model.predict(X_windows_test)
        mean_test = pred_test[..., 0].flatten()
        scale_test = pred_test[..., 1].flatten()

        full_mean_test = np.full(n_test, 0.0, dtype=np.float64)
        full_scale_test = np.full(n_test, 0.0, dtype=np.float64)
        full_mean_test[window:] = mean_test.astype(np.float64)
        full_scale_test[window:] = scale_test.astype(np.float64)

        test_h_updated = test_h.with_columns([
            pl.Series(f'bnn_mean_h{h}', full_mean_test).cast(pl.Float64),
            pl.Series(f'bnn_scale_h{h}', full_scale_test).cast(pl.Float64)
        ])
        
        test_full = test_full.filter(pl.col('horizon') != h)
        test_full = pl.concat([test_full, test_h_updated])

    print(f"  Generated bnn_mean_h{h} and bnn_scale_h{h}")

print("\n✅ BNN training complete")
print("="*60)
##%%


BNN TRAINING - FEATURE A + ULTRA FEATURES

--- HORIZON 1 ---
  Target: mean=-0.083, std=11.700

🔍 DEBUG BNN TRAINING DATA H=1:
  X_train shape: (1394643, 10, 4)
  y_train shape: (1394643,)
  X_train min/max: [-13.488447, 14.103717]
  y_train min/max: [-73.073814, 88.352036]
  X_train NaN count: 0
  y_train NaN count: 0


I0000 00:00:1775570787.575056      24 gpu_device.cc:2019] Created device /job:localhost/replica:0/task:0/device:GPU:0 with 13757 MB memory:  -> device: 0, name: Tesla T4, pci bus id: 0000:00:04.0, compute capability: 7.5
I0000 00:00:1775570787.581290      24 gpu_device.cc:2019] Created device /job:localhost/replica:0/task:0/device:GPU:1 with 13757 MB memory:  -> device: 1, name: Tesla T4, pci bus id: 0000:00:05.0, compute capability: 7.5


Epoch 1/20


I0000 00:00:1775570791.500345      95 service.cc:152] XLA service 0x7a9ed5e40490 initialized for platform CUDA (this does not guarantee that XLA will be used). Devices:
I0000 00:00:1775570791.500379      95 service.cc:160]   StreamExecutor device (0): Tesla T4, Compute Capability 7.5
I0000 00:00:1775570791.500383      95 service.cc:160]   StreamExecutor device (1): Tesla T4, Compute Capability 7.5
I0000 00:00:1775570791.594944      95 cuda_dnn.cc:529] Loaded cuDNN version 91002
I0000 00:00:1775570791.770603      95 device_compiler.h:188] Compiled cluster using XLA!  This line is logged at most once for the lifetime of the process.


10896/10896 [==============================] - 36s 3ms/step - loss: -0.0046
Epoch 2/20
10896/10896 [==============================] - 32s 3ms/step - loss: -0.0651
Epoch 3/20
10896/10896 [==============================] - 32s 3ms/step - loss: -0.1622
Epoch 4/20
10896/10896 [==============================] - 32s 3ms/step - loss: -0.3614
Epoch 5/20
10896/10896 [==============================] - 32s 3ms/step - loss: -0.4237
Epoch 6/20
10896/10896 [==============================] - 32s 3ms/step - loss: -0.4578
Epoch 7/20
10896/10896 [==============================] - 31s 3ms/step - loss: -0.4433
Epoch 8/20
10896/10896 [==============================] - 32s 3ms/step - loss: -0.4701
Epoch 9/20
10896/10896 [==============================] - 32s 3ms/step - loss: -0.4759
Epoch 10/20
10896/10896 [==============================] - 32s 3ms/step - loss: -0.4824
Epoch 11/20
10896/10896 [==============================] - 31s 3ms/step - loss: -0.4530
Epoch 12/20
10896/10896 [===========================

In [5]:
# Po wytrenowaniu modelu dla każdego horyzontu
import pickle
from pathlib import Path

# Utwórz folder dla modeli
Path('bnn_models').mkdir(exist_ok=True)

# Słownik do przechowywania modeli
bnn_models = {}

for h in horizons:
    # ... twój kod treningowy ...
    
    # Zapisz model po treningu
    model.save(f'bnn_models/bnn_model_h{h}.keras')
    
    # Opcjonalnie: zapisz też w słowniku
    bnn_models[h] = model

# Zapisz wszystkie modele jako jeden plik
with open('bnn_models/all_models.pkl', 'wb') as f:
    pickle.dump(bnn_models, f)

In [6]:
# Po wygenerowaniu predykcji, zapisz je do pliku
import pickle
from pathlib import Path

# Przygotuj dane do zapisu
bnn_predictions = {
    'train': {},
    'test': {}
}

for h in horizons:
    train_h = train_full.filter(pl.col('horizon') == h)
    test_h = test_full.filter(pl.col('horizon') == h)
    
    # POPRAWA: dodaj 'f' przed stringiem
    bnn_predictions['train'][h] = {
        'mean': train_h[f'bnn_mean_h{h}'].to_numpy(),  # f-string
        'scale': train_h[f'bnn_scale_h{h}'].to_numpy()  # f-string
    }
    bnn_predictions['test'][h] = {
        'mean': test_h[f'bnn_mean_h{h}'].to_numpy(),    # f-string
        'scale': test_h[f'bnn_scale_h{h}'].to_numpy()   # f-string
    }

# Zapisz do pliku
with open('bnn_predictions.pkl', 'wb') as f:
    pickle.dump(bnn_predictions, f)

In [7]:
##%%
# ============================================
# CORRELATION ANALYSIS - FEATURE A BNN
# ============================================

print("\n" + "="*60)
print("BNN CORRELATION ANALYSIS")
print("="*60)

for h in horizons:
    print(f"\n--- HORIZON {h} ---")
    
    train_h = train_full.filter(pl.col('horizon') == h)
    
    if len(train_h) == 0:
        print(f"  Brak danych dla H={h}")
        continue
    
    # BNN correlations
    bnn_cols = [f'bnn_mean_h{h}', f'bnn_scale_h{h}']
    
    for col in bnn_cols:
        if col in train_h.columns:
            corr = train_h.select(
                pl.corr('y_target', col)
            ).item()
            
            mean_val = train_h.select(pl.mean(col)).item()
            std_val = train_h.select(pl.std(col)).item()
            
            print(f"  {col}:")
            print(f"    Korelacja z target: {corr:.4f}")
            print(f"    Mean: {mean_val:.4f}, Std: {std_val:.4f}")
            
            if abs(corr) < 0.01:
                interpretation = "❌ Brak korelacji (szum)"
            elif abs(corr) < 0.05:
                interpretation = "⚠️ Bardzo słaba korelacja"
            elif abs(corr) < 0.1:
                interpretation = "🟡 Słaba korelacja"
            elif abs(corr) < 0.2:
                interpretation = "🟢 Umiarkowana korelacja"
            else:
                interpretation = "✅ Silna korelacja"
            
            print(f"    Interpretacja: {interpretation}")
    
    # Feature A correlation
    if 'feature_a' in train_h.columns:
        feature_a_corr = train_h.select(
            pl.corr('y_target', 'feature_a')
        ).item()
        print(f"  feature_a: Korelacja z target: {feature_a_corr:.4f}")

print("\n" + "="*60)
#%%


BNN CORRELATION ANALYSIS

--- HORIZON 1 ---
  bnn_mean_h1:
    Korelacja z target: 0.0061
    Mean: -0.0019, Std: 0.0024
    Interpretacja: ❌ Brak korelacji (szum)
  bnn_scale_h1:
    Korelacja z target: -0.0068
    Mean: 0.1867, Std: 0.1974
    Interpretacja: ❌ Brak korelacji (szum)
  feature_a: Korelacja z target: 0.0057

--- HORIZON 3 ---
  bnn_mean_h3:
    Korelacja z target: 0.0054
    Mean: -0.0102, Std: 0.0197
    Interpretacja: ❌ Brak korelacji (szum)
  bnn_scale_h3:
    Korelacja z target: -0.0132
    Mean: 0.2288, Std: 0.2531
    Interpretacja: ⚠️ Bardzo słaba korelacja
  feature_a: Korelacja z target: 0.0097

--- HORIZON 10 ---
  bnn_mean_h10:
    Korelacja z target: -0.0034
    Mean: 0.0014, Std: 0.0096
    Interpretacja: ❌ Brak korelacji (szum)
  bnn_scale_h10:
    Korelacja z target: -0.0191
    Mean: 0.1777, Std: 0.2025
    Interpretacja: ⚠️ Bardzo słaba korelacja
  feature_a: Korelacja z target: 0.0140

--- HORIZON 25 ---
  bnn_mean_h25:
    Korelacja z target: -0.0196

In [8]:
# ============================================
# PREPARE ALL FEATURES FOR LIGHTGBM
# ============================================

# Get all feature_* columns for LightGBM
all_feature_cols = [c for c in train_full.columns if c.startswith('feature_')]
print(f"All feature_* columns: {len(all_feature_cols)}")

# Add BNN features
bnn_feature_cols = []
for h in horizons:
    bnn_feature_cols.extend([f'bnn_mean_h{h}', f'bnn_scale_h{h}'])

# Add time features if they exist (check first to avoid duplicates)
time_feature_names = []
for col in ['time_mod_50', 'time_mod_200', 'time_mod_7', 'ts_ema_5']:
    if col in train_full.columns:
        time_feature_names.append(col)

# Final feature list (no duplicates)
final_features = all_feature_cols + time_feature_names + bnn_feature_cols

print(f"\n📊 Feature breakdown:")
print(f"  Base features: {len(all_feature_cols)}")
print(f"  Time features: {len(time_feature_names)}")
print(f"  BNN features: {len(bnn_feature_cols)}")
print(f"  Total: {len(final_features)}")

# Verify all features exist
missing_features = [f for f in final_features if f not in train_full.columns]
if missing_features:
    print(f"\n⚠️ Missing features: {missing_features[:10]}...")
else:
    print(f"\n✅ All features available")

# ============================================
# ADD TIME FEATURES IF MISSING (BEFORE LIGHTGBM)
# ============================================

time_features_to_add = []

if 'time_mod_50' not in train_full.columns:
    time_features_to_add.append(('time_mod_50', pl.col('ts_index') % 50))
if 'time_mod_200' not in train_full.columns:
    time_features_to_add.append(('time_mod_200', pl.col('ts_index') % 200))
if 'time_mod_7' not in train_full.columns:
    time_features_to_add.append(('time_mod_7', pl.col('ts_index') % 7))
if 'ts_ema_5' not in train_full.columns:
    time_features_to_add.append(('ts_ema_5', pl.col('ts_index').rolling_mean(window_size=5)))

if time_features_to_add:
    for name, expr in time_features_to_add:
        train_full = train_full.with_columns(expr.alias(name))
        test_full = test_full.with_columns(expr.alias(name))
    print(f"✅ Added time features: {[name for name, _ in time_features_to_add]}")
    # Update final_features
    new_time_features = [name for name, _ in time_features_to_add]
    final_features = list(dict.fromkeys(final_features + new_time_features))
else:
    print("✅ Time features already exist")

print(f"Total features now: {len(final_features)}")

# ============================================
# METRICS FUNCTION
# ============================================

def _clip01(x: float) -> float:
    return float(np.minimum(np.maximum(x, 0.0), 1.0))

def weighted_rmse_score(y_target, y_pred, w) -> float:
    denom = np.sum(w * y_target ** 2)
    if denom == 0:
        return 0.0
    ratio = np.sum(w * (y_target - y_pred) ** 2) / denom
    clipped = _clip01(ratio)
    val = 1.0 - clipped
    return float(np.sqrt(val))

# ============================================
# LIGHTGBM TRAINING WITH BNN FEATURES + TIME FEATURES
# ============================================

print("\n" + "="*60)
print("LIGHTGBM TRAINING WITH FEATURE A BNN + TIME FEATURES")
print("="*60)

def get_lgb_params(horizon: int) -> dict:
    """Optuna-optimized dla BNN features + regime patterns"""
    best_params = {
        "1": {"learning_rate": 0.02855, "num_leaves": 240, "min_child_samples": 77, 
              "lambda_l2": 3.89, "feature_fraction": 0.809, "bagging_fraction": 0.893, "bagging_freq": 3},
        "3": {"learning_rate": 0.02705, "num_leaves": 233, "min_child_samples": 62, 
              "lambda_l2": 6.22, "feature_fraction": 0.832, "bagging_fraction": 0.908, "bagging_freq": 3},
        "10": {"learning_rate": 0.02503, "num_leaves": 183, "min_child_samples": 162, 
               "lambda_l2": 11.92, "feature_fraction": 0.732, "bagging_fraction": 0.856, "bagging_freq": 2},
        "25": {"learning_rate": 0.02062, "num_leaves": 343, "min_child_samples": 278, 
               "lambda_l2": 24.86, "feature_fraction": 0.622, "bagging_fraction": 0.846, "bagging_freq": 4}
    }
    
    base = {
        'objective': 'regression', 'metric': 'rmse',
        'random_state': 42, 'verbose': -1, 'n_estimators': 2000
    }
    
    params = {**base, **best_params[str(horizon)]}
    return params

predictions_by_horizon = {}
scores_by_horizon = {}
all_metrics = {}

for horizon in horizons:
    print(f"\n{'='*60}")
    print(f"HORIZON: {horizon}")
    print(f"{'='*60}")

    # Filter data by horizon
    train_h = train_full.filter(pl.col('horizon') == horizon)

    # Prepare features (final_features already includes time features)
    X_h = train_h.select(final_features).to_numpy()
    y_h = train_h.select("y_target").to_numpy().ravel()
    w_h = train_h.select("weight").to_numpy().ravel()

    print(f"Train samples: {len(train_h):,}")
    print(f"Features: {X_h.shape[1]}")

    # Train model with per-horizon parameters
    start_time = time.time()
    
    params = get_lgb_params(horizon)
    print(f"Params: num_leaves={params['num_leaves']}, n_estimators={params['n_estimators']}")
    
    model = lgb.LGBMRegressor(**params)
    model.fit(X_h, y_h, sample_weight=w_h)

    train_time = time.time() - start_time

    # Evaluate
    y_pred_h = model.predict(X_h)
    score_h = weighted_rmse_score(y_h, y_pred_h, w_h)
    pearson = np.corrcoef(y_h, y_pred_h)[0, 1]
    mae = np.mean(np.abs(y_h - y_pred_h))
    r2 = 1 - np.sum((y_h - y_pred_h)**2) / np.sum((y_h - np.mean(y_h))**2)

    print(f"\n📊 TRAINING METRICS:")
    print(f"  Weighted RMSE: {score_h:.6f}")
    print(f"  Pearson: {pearson:.6f}")
    print(f"  MAE: {mae:.6f}")
    print(f"  R²: {r2:.6f}")
    print(f"  y_pred std: {y_pred_h.std():.4f} (target std: {y_h.std():.4f})")
    print(f"  Training time: {train_time:.2f}s")

    # Store metrics
    all_metrics[horizon] = {
        'train_score': score_h,
        'pearson': pearson,
        'mae': mae,
        'r2': r2,
        'pred_std': y_pred_h.std(),
        'target_std': y_h.std(),
        'train_time': train_time
    }
    scores_by_horizon[horizon] = score_h

    # Feature importance
    feature_importance = list(zip(final_features, model.feature_importances_))
    feature_importance.sort(key=lambda x: x[1], reverse=True)
    
    print(f"\n🎯 TOP 10 FEATURES:")
    for feat, imp in feature_importance[:10]:
        print(f"  {feat}: {imp}")

    # Test predictions
    test_h = test_full.filter(pl.col('horizon') == horizon)
    X_test_h = test_h.select(final_features).fill_null(0).to_numpy()

    preds_h = model.predict(X_test_h)

    print(f"\n📊 TEST PREDICTIONS:")
    print(f"  Samples: {len(preds_h):,}")
    print(f"  Mean: {np.mean(preds_h):.6f}")
    print(f"  Std: {np.std(preds_h):.6f}")
    print(f"  Range: [{np.min(preds_h):.4f}, {np.max(preds_h):.4f}]")

    predictions_by_horizon[horizon] = {
        'ids': test_h.select('id').to_numpy().ravel(),
        'predictions': preds_h
    }

# ============================================
# COMBINE PREDICTIONS
# ============================================
print("\n" + "="*60)
print("COMBINING PREDICTIONS")
print("="*60)

all_ids = []
all_preds = []

for h in horizons:
    all_ids.extend(predictions_by_horizon[h]['ids'])
    all_preds.extend(predictions_by_horizon[h]['predictions'])

submission_df = pl.DataFrame({'id': all_ids, 'prediction': all_preds})

# ============================================
# SAVE SUBMISSION
# ============================================
print("\n" + "="*60)
print("SAVING SUBMISSION")
print("="*60)

timestamp = datetime.datetime.now().strftime("%Y%m%d_%H%M%S")
submission_filename = f"lightgbm_bnn_time_features_{timestamp}.csv"
submission_path = Path(output_dir) / submission_filename
submission_df.write_csv(submission_path)

print(f"✅ Submission saved: {submission_path}")
print(f"📊 File size: {submission_path.stat().st_size / 1024**2:.2f} MB")
print(f"📝 Records: {len(submission_df)}")

# ============================================
# FINAL SUMMARY
# ============================================
print("\n" + "="*80)
print("FINAL SUMMARY")
print("="*80)

print(f"\n{'Horizon':<8} {'Score':<12} {'Pearson':<10} {'MAE':<10} {'R²':<10} {'Time(s)':<10}")
print("-" * 65)
for h in horizons:
    m = all_metrics[h]
    print(f"{h:<8} {m['train_score']:<12.6f} {m['pearson']:<10.6f} {m['mae']:<10.6f} {m['r2']:<10.6f} {m['train_time']:<10.2f}")

print(f"\nFeatures: {len(final_features)} (feature_* + time_features + BNN)")
print(f"Time features: {time_feature_names}")
print(f"Test predictions range: [{np.min(all_preds):.4f}, {np.max(all_preds):.4f}]")
print(f"Submission: {submission_path}")
print("="*80)

print("\n✅ Notebook complete – ready for Kaggle submission!")

All feature_* columns: 86

📊 Feature breakdown:
  Base features: 86
  Time features: 0
  BNN features: 8
  Total: 94

✅ All features available
✅ Added time features: ['time_mod_50', 'time_mod_200', 'time_mod_7', 'ts_ema_5']
Total features now: 98

LIGHTGBM TRAINING WITH FEATURE A BNN + TIME FEATURES

HORIZON: 1
Train samples: 1,394,653
Features: 98
Params: num_leaves=240, n_estimators=2000

📊 TRAINING METRICS:
  Weighted RMSE: 0.549266
  Pearson: 0.221008
  MAE: 2.087838
  R²: 0.021596
  y_pred std: 0.6544 (target std: 11.6997)
  Training time: 504.27s

🎯 TOP 10 FEATURES:
  feature_al: 24586
  feature_c: 12337
  feature_g: 12260
  feature_f: 12258
  feature_b: 12240
  feature_d: 12127
  feature_e: 11905
  bnn_mean_h1: 11166
  feature_s: 10786
  bnn_scale_h1: 10473

📊 TEST PREDICTIONS:
  Samples: 379,617
  Mean: -0.112232
  Std: 0.635162
  Range: [-11.5519, 4.2544]

HORIZON: 3
Train samples: 1,385,816
Features: 98
Params: num_leaves=233, n_estimators=2000

📊 TRAINING METRICS:
  Weighted